In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="alexandrainst/nst-da", 
    repo_type="dataset", local_dir="./nst-da", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 129 files: 100%|██████████| 129/129 [00:45<00:00,  2.85it/s]


'/home/ubuntu/nst-da'

In [3]:
files = glob('nst-da/*/*.parquet')
len(files)

129

In [4]:
df = pd.read_parquet(files[0])
df

,audio,text,speaker_id,age,sex,dialect,recording_datetime
0,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,Der har været komiske indslag i tragedierne.,363,27,Male,Sønderjylland,2000-07-26T10:44:27
1,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,konkurrenter,363,27,Male,Sønderjylland,2000-07-26T10:44:27
2,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,Det ville kun tage nogle få sekunder at løbe t...,363,27,Male,Sønderjylland,2000-07-26T10:44:27
3,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,"Materialet, du producerer, kan du vedlægge din...",363,27,Male,Sønderjylland,2000-07-26T10:44:27
4,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,Gaden er lukket for gennemgående færdsel ti må...,363,27,Male,Sønderjylland,2000-07-26T10:44:27
...,...,...,...,...,...,...,...
1640,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,Hendes navn er stavet forkert i bladet.,393,46,Male,Vestjylland,2000-08-15T15:14:55
1641,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,sidste måned,393,46,Male,Vestjylland,2000-08-15T15:14:55
1642,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,fredag,393,46,Male,Vestjylland,2000-08-15T15:14:55
1643,{'bytes': b'NIST_1A\n 1024\nsample_coding -s...,"Adskillige af de huse, som kommunen byggede si...",393,46,Male,Vestjylland,2000-08-15T15:14:55


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{int(df['speaker_id'].iloc[i])}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 6/6 [06:07<00:00, 61.26s/it]


In [7]:
len(data)

237330

In [8]:
with open('nst-da.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('nst-da-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'nst-da_audio/nst-da-data-train-00095-of-00111-0c6edf6b791abad8_0.mp3',
 'text': 'Der har været komiske indslag i tragedierne.',
 'speaker': 'nst-da_audio_363'}

In [11]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'nst-da')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.93ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  87%|████████▋ | 9.36MB / 10.7MB, 46.7MB/s  
Processing Files (1 / 1): 100%|██████████| 10.7MB / 10.7MB, 26.8MB/s  
Processing Files (1 / 1): 100%|██████████| 10.7MB / 10.7MB, 17.9MB/s  
New Data Upload: 100%|██████████| 10.7MB / 10.7MB, 17.9MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/7475264663f035e5fd25207866016282995c9154', commit_message='Upload dataset', commit_description='', oid='7475264663f035e5fd25207866016282995c9154', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [15]:
# !zip -rq nst-da_audio.zip nst-da_audio

In [16]:
# !hf upload malaysia-ai/Multilingual-TTS nst-da_audio.zip --repo-type=dataset

In [19]:
# !zip -rq nst-da_audio_neucodec.zip nst-da_audio_neucodec

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS nst-da_audio_neucodec.zip --repo-type=dataset